# Network Traffic Pattern Discovery Using Unsupervised Learning
### UNSW-NB15 | K-Means · Agglomerative Clustering · DBSCAN

## 1. Project Overview

This notebook applies unsupervised machine learning to network-flow records from the **UNSW-NB15** dataset. The objective is to group traffic records with similar numerical characteristics and then examine whether the resulting groups differ in their known normal/attack labels.

**Research question:** Can clustering algorithms discover meaningful structure in network traffic without using attack labels during model fitting?

### Objectives
- Inspect and prepare a working sample of UNSW-NB15.
- Clean missing and infinite numeric values.
- Standardize numerical features.
- Explore clusters with K-Means, Agglomerative Clustering, and DBSCAN.
- Compare cluster separation using the silhouette coefficient.
- Visualize K-Means clusters with PCA.
- Interpret clusters using labels only after clustering.

> **Important:** This is an exploratory clustering study, not a validated intrusion-detection system. A cluster is not automatically equivalent to an attack class.

## 2. Dataset and Setup

### Dataset

- **Dataset:** UNSW-NB15
- **Working sample:** Up to 5,000 records from each of the four supplied data files (up to 20,000 records total).
- **Feature description file:** `NUSW-NB15_features.csv` (use the exact filename supplied with your dataset).
- **Target/interpretation columns:** `attack_cat` and `Label`.

The four CSV files and feature-description CSV must be available in the notebook runtime. The code below uses Google Colab paths under `/content/`.

### Expected files

```text
/content/
├── NUSW-NB15_features.csv
├── UNSW-NB15_1.csv
├── UNSW-NB15_2.csv
├── UNSW-NB15_3.csv
└── UNSW-NB15_4.csv
```

Upload or mount the dataset files before running the data-loading cells. The dataset itself is not embedded in this notebook.

## 3. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

## 4. Load Feature Names and Network Records

In [ ]:
# Load the feature-name file.
# Keep the filename exactly as it appears in your downloaded dataset.
features_path = "/content/NUSW-NB15_features.csv"
features_df = pd.read_csv(features_path, encoding="latin1")

column_names = features_df["Name"].astype(str).tolist()

print("Number of feature names:", len(column_names))
print("First 10 feature names:", column_names[:10])

In [ ]:
# Read up to 5,000 rows from each of the four data partitions.
data_files = [
    "/content/UNSW-NB15_1.csv",
    "/content/UNSW-NB15_2.csv",
    "/content/UNSW-NB15_3.csv",
    "/content/UNSW-NB15_4.csv",
]

parts = []
for file_path in data_files:
    part = pd.read_csv(file_path, header=None, nrows=5000)
    if part.shape[1] != len(column_names):
        raise ValueError(
            f"{file_path} has {part.shape[1]} columns, "
            f"but the feature file lists {len(column_names)} names."
        )
    parts.append(part)

df = pd.concat(parts, ignore_index=True)
df.columns = column_names

print("Working sample shape:", df.shape)
display(df.head())

## 5. Initial Data Inspection

Check the dimensions, data types, missing values, duplicate rows, and label columns. The labels are retained for later interpretation, but they will not be supplied to the clustering algorithms.

In [ ]:
print("Dataset shape:", df.shape)

print("\nData types:")
display(df.dtypes.value_counts().rename("Column count").to_frame())

print("\nTotal missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

print("\nLabel columns present:")
print([col for col in ["attack_cat", "Label"] if col in df.columns])

display(df[["attack_cat", "Label"]].head())

## 6. Exploratory Data Analysis

Inspect the distribution of the binary label and the attack categories. These columns are used only to understand the sample and later interpret clusters; they are excluded from the model features.

In [ ]:
if "Label" in df.columns:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x="Label")
    plt.title("Distribution of Normal and Attack Labels")
    plt.xlabel("Label (0 = normal, 1 = attack)")
    plt.ylabel("Record count")
    plt.tight_layout()
    plt.show()

if "attack_cat" in df.columns:
    plt.figure(figsize=(10, 5))
    order = df["attack_cat"].fillna("Missing").value_counts().index
    sns.countplot(
        data=df.assign(attack_cat=df["attack_cat"].fillna("Missing")),
        y="attack_cat",
        order=order
    )
    plt.title("Distribution of Attack Categories")
    plt.xlabel("Record count")
    plt.ylabel("Attack category")
    plt.tight_layout()
    plt.show()

## 7. Prepare Features for Clustering

### Feature selection

- Remove source and destination IP addresses because they are identifiers.
- Remove `attack_cat` and `Label` to avoid using the known answers during unsupervised learning.
- Keep numerical features for this clustering experiment.
- Replace infinite values with missing values, impute missing values with the median, and remove constant columns.

This notebook deliberately uses **numerical features only**. Categorical variables such as protocol and service are not one-hot encoded in this version.

In [ ]:
# Save known labels for post-clustering interpretation.
attack_categories = df["attack_cat"].copy() if "attack_cat" in df.columns else None
attack_labels = df["Label"].copy() if "Label" in df.columns else None

# Remove identifiers and target/label columns before clustering.
excluded_columns = ["srcip", "dstip", "attack_cat", "Label"]
X_numeric = (
    df.drop(columns=excluded_columns, errors="ignore")
      .select_dtypes(include=np.number)
      .copy()
)

# Replace positive/negative infinity with missing values.
X_numeric = X_numeric.replace([np.inf, -np.inf], np.nan)

# Median imputation.
imputer = SimpleImputer(strategy="median")
X_numeric = pd.DataFrame(
    imputer.fit_transform(X_numeric),
    columns=X_numeric.columns,
    index=df.index
)

# Remove constant columns; they cannot help distinguish clusters.
constant_columns = X_numeric.columns[X_numeric.nunique() <= 1].tolist()
X_numeric = X_numeric.drop(columns=constant_columns)

print("Numeric feature matrix shape:", X_numeric.shape)
print("Removed constant columns:", constant_columns)
print("Remaining missing values:", int(X_numeric.isna().sum().sum()))

## 8. Feature Scaling

Standardization puts numerical features on a comparable scale. This is important for distance-based methods such as K-Means, Agglomerative Clustering, and DBSCAN.

The scaler is fitted to the working sample used in this exploratory notebook.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numeric)

print("Scaled matrix shape:", X_scaled.shape)
print("All values finite:", bool(np.isfinite(X_scaled).all()))

## 9. Select the Number of K-Means Clusters

The elbow method plots K-Means inertia for several values of *k*. Look for a point where the reduction in inertia begins to slow. The elbow is a heuristic, not proof that a particular number of clusters is correct.

In [ ]:
k_values = range(2, 7)
inertia_values = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(k_values), inertia_values, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.tight_layout()
plt.show()

## 10. K-Means Clustering

For the main cluster interpretation, K-Means is fitted with **two clusters** on the prepared numerical feature matrix. The choice of two clusters is an experimental setting for this case study, not a claim that network traffic naturally contains only two groups.

In [ ]:
kmeans_model = KMeans(
    n_clusters=2,
    random_state=RANDOM_STATE,
    n_init=10
)
kmeans_labels = kmeans_model.fit_predict(X_scaled)

df_clustered = df.copy()
df_clustered["Cluster"] = kmeans_labels

print("K-Means cluster sizes:")
display(df_clustered["Cluster"].value_counts().sort_index().rename("Records").to_frame())

kmeans_silhouette = silhouette_score(
    X_scaled,
    kmeans_labels,
    sample_size=min(3000, len(X_scaled)),
    random_state=RANDOM_STATE
)
print(f"K-Means silhouette score (sampled evaluation): {kmeans_silhouette:.4f}")
print(f"K-Means inertia: {kmeans_model.inertia_:.2f}")

## 11. Interpret K-Means Clusters Using Known Labels

The labels are added back **after** clustering. Crosstabs show the proportion of normal/attack records and attack categories in each cluster. This is post-hoc interpretation, not supervised training.

In [ ]:
if attack_categories is not None:
    df_clustered["attack_cat"] = attack_categories.values
    print("Attack-category composition within each cluster:")
    display(
        pd.crosstab(
            df_clustered["Cluster"],
            df_clustered["attack_cat"].fillna("Missing"),
            normalize="index"
        ).round(3)
    )

if attack_labels is not None:
    df_clustered["Label"] = attack_labels.values
    print("Normal/attack composition within each cluster:")
    display(
        pd.crosstab(
            df_clustered["Cluster"],
            df_clustered["Label"],
            normalize="index"
        ).round(3)
    )

## 12. Visualize Clusters Using PCA

Principal Component Analysis (PCA) projects the standardized numerical features into two dimensions for visualization. The plot is a compressed view of the data; it does not preserve all information from the original feature space.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 5))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=kmeans_labels,
    cmap="viridis",
    s=8,
    alpha=0.6
)
plt.title("K-Means Clusters of UNSW-NB15 Records (PCA Projection)")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.tight_layout()
plt.show()

explained_variance = pca.explained_variance_ratio_.sum()
print(f"Variance explained by the first two components: {explained_variance:.3f}")

## 13. Agglomerative Clustering

Agglomerative Clustering builds a hierarchy of clusters and then returns the requested number of groups. To keep the comparison computationally manageable, it is evaluated on the same fixed-size sample used for model comparison below.

In [ ]:
# Draw one reproducible evaluation sample for fair model comparison.
sample_size = min(3000, len(X_scaled))
eval_indices = np.random.RandomState(RANDOM_STATE).choice(
    len(X_scaled), size=sample_size, replace=False
)
X_eval = X_scaled[eval_indices]

agg_model = AgglomerativeClustering(n_clusters=2)
agg_eval_labels = agg_model.fit_predict(X_eval)

print("Agglomerative cluster sizes:")
display(pd.Series(agg_eval_labels).value_counts().sort_index().rename("Records").to_frame())

agg_silhouette = silhouette_score(X_eval, agg_eval_labels)
print(f"Agglomerative silhouette score: {agg_silhouette:.4f}")

## 14. DBSCAN Clustering

DBSCAN groups dense regions and labels records that do not belong to a dense region as noise (`-1`). Its result depends strongly on `eps` and `min_samples`; the values below are starting parameters for this experiment, not optimized settings.

In [ ]:
dbscan_model = DBSCAN(
    eps=3.0,
    min_samples=10
)
dbscan_eval_labels = dbscan_model.fit_predict(X_eval)

dbscan_cluster_ids = set(dbscan_eval_labels)
dbscan_n_clusters = len(dbscan_cluster_ids - {-1})
dbscan_n_noise = int(np.sum(dbscan_eval_labels == -1))

print("DBSCAN clusters (excluding noise):", dbscan_n_clusters)
print("DBSCAN noise records:", dbscan_n_noise)
print("\nDBSCAN label counts (-1 represents noise):")
display(
    pd.Series(dbscan_eval_labels)
      .value_counts()
      .sort_index()
      .rename("Records")
      .to_frame()
)

## 15. Compare Clustering Algorithms

The silhouette coefficient measures how similar a record is to its own cluster compared with other clusters. A higher value generally indicates better separation, but it does not establish that clusters correspond to real attack classes.

K-Means, Agglomerative Clustering, and DBSCAN are evaluated on the same `X_eval` sample. For DBSCAN, noise records are excluded from the silhouette calculation; this difference must be considered when interpreting the comparison.

In [ ]:
comparison_results = {
    "K-Means": silhouette_score(X_eval, KMeans(
        n_clusters=2,
        random_state=RANDOM_STATE,
        n_init=10
    ).fit_predict(X_eval)),
    "Agglomerative": silhouette_score(X_eval, agg_eval_labels),
}

# Silhouette score is defined only when at least two non-noise clusters remain.
dbscan_non_noise = dbscan_eval_labels != -1
dbscan_remaining_clusters = set(dbscan_eval_labels[dbscan_non_noise])

if dbscan_non_noise.sum() > 1 and len(dbscan_remaining_clusters) >= 2:
    comparison_results["DBSCAN (noise excluded)"] = silhouette_score(
        X_eval[dbscan_non_noise],
        dbscan_eval_labels[dbscan_non_noise]
    )
else:
    comparison_results["DBSCAN (noise excluded)"] = np.nan

comparison_df = (
    pd.DataFrame(
        comparison_results.items(),
        columns=["Algorithm", "Silhouette Score"]
    )
    .sort_values("Silhouette Score", ascending=False, na_position="last")
    .reset_index(drop=True)
)

display(comparison_df.round(4))

In [ ]:
plt.figure(figsize=(8, 4))
plot_df = comparison_df.dropna(subset=["Silhouette Score"])
sns.barplot(
    data=plot_df,
    x="Algorithm",
    y="Silhouette Score"
)
plt.title("Clustering Algorithm Comparison")
plt.ylabel("Silhouette score (higher is generally better)")
plt.xlabel("")
plt.ylim(min(0, plot_df["Silhouette Score"].min() - 0.05), 1)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 16. Cluster Profiling

Compare the mean values of selected numerical features across the K-Means clusters. These summaries can help describe how the groups differ. They should be interpreted in the context of the feature definitions and the sampling procedure.

In [ ]:
profile = (
    df_clustered.groupby("Cluster")[X_numeric.columns]
    .mean()
    .T
)

display(profile.round(3))

## 17. Results and Discussion

Run the notebook from top to bottom to generate the scores for the current dataset files and sample. The exact results can vary if the dataset files, sample, preprocessing, or algorithm parameters change.

When discussing the results, report:
- The silhouette score for each algorithm.
- The number and size of clusters found.
- The number of DBSCAN noise records.
- The normal/attack composition of K-Means clusters.
- The proportion of variance represented by the first two PCA components.

### Interpretation cautions

1. A higher silhouette score indicates better geometric separation under the chosen distance metric; it does not prove better intrusion detection.
2. K-Means and Agglomerative Clustering are forced to assign every record to a cluster. DBSCAN can mark records as noise.
3. DBSCAN's score is calculated after excluding noise records, so it is not a perfectly identical comparison to the other two scores.
4. The working sample takes the first 5,000 rows from each file. It may not represent the full dataset distribution.
5. Only numerical columns are used in this notebook. Categorical fields such as protocol, state, and service are not included in the clustering features.

## 18. Conclusion

This case study applies three unsupervised learning algorithms—K-Means, Agglomerative Clustering, and DBSCAN—to a working sample of UNSW-NB15 network traffic records. The workflow includes data inspection, numeric feature preparation, missing-value imputation, feature scaling, clustering, PCA visualization, silhouette-based comparison, and post-hoc interpretation using known labels.

The results should be treated as exploratory evidence of structure in the selected network-flow features. Further work would be required before using these clusters in a practical security-monitoring or intrusion-detection pipeline.

## 19. Possible Future Improvements

- Evaluate the complete dataset rather than a limited sample.
- Use stratified or randomized sampling to improve representativeness.
- Tune DBSCAN parameters systematically.
- Test alternative distance metrics and clustering algorithms.
- Explore carefully encoded categorical features.
- Compare cluster assignments with labels using external metrics such as adjusted Rand index, while keeping labels out of model fitting.
- Validate results on a separate dataset or time period.

## References and Attribution

- **Dataset:** UNSW-NB15 network traffic dataset. Use the original dataset distribution and accompanying feature-description file.
- **Libraries:** pandas, NumPy, Matplotlib, Seaborn, and scikit-learn.

**Reproducibility note:** This notebook expects the dataset CSV files to be supplied separately. No dataset records are bundled in this notebook.